In [ ]:
# Original methods
import math

# Parameters
S0 = 100.0  # spot price
K = 100.0   # strike
r = 0.05    # risk-free rate
sigma = 0.2 # volatility
T = 1.0     # maturity (years)

# Standard normal CDF
def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

# 1) Black–Scholes closed-form formula for European call
def bs_call_price(S, K, r, sigma, T):
    if T <= 0 or sigma <= 0:
        return max(S - K * math.exp(-r * T), 0.0)
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return S * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)

bs_price = bs_call_price(S0, K, r, sigma, T)

# 2) Cox–Ross–Rubinstein binomial tree for European call
def cr_tree_call(S0, K, r, sigma, T, N):
    dt = T / N
    u = math.exp(sigma * math.sqrt(dt))
    d = 1.0 / u
    disc = math.exp(-r * dt)
    p = (math.exp(r * dt) - d) / (u - d)

    # terminal stock prices
    prices = [S0 * (u**j) * (d**(N - j)) for j in range(N + 1)]
    # terminal option values
    values = [max(price - K, 0.0) for price in prices]

    # backward induction
    for i in range(N - 1, -1, -1):
        values = [disc * (p * values[j + 1] + (1 - p) * values[j]) for j in range(i + 1)]

    return values[0]

binom_price_200 = cr_tree_call(S0, K, r, sigma, T, N=200)

# 3) Explicit finite-difference scheme for Black–Scholes PDE (European call)
def fd_explicit_call(S0, K, r, sigma, T,
                     S_max_factor=3.0, M=200, N=4000):
    # S-grid: 0 ... S_max
    S_max = S_max_factor * S0
    dS = S_max / M
    dt = T / N

    S_vals = [i * dS for i in range(M + 1)]

    # terminal payoff at maturity
    V = [max(S - K, 0.0) for S in S_vals]

    # time stepping backwards
    for n in range(N - 1, -1, -1):
        t = n * dt
        V_new = V.copy()

        # boundary conditions for a call
        V_new[0] = 0.0
        V_new[M] = S_max - K * math.exp(-r * (T - t))

        for i in range(1, M):
            # coefficients using S_i = i * dS
            a = 0.5 * dt * (sigma**2 * i**2 - r * i)
            b = 1.0 - dt * (sigma**2 * i**2 + r)
            c = 0.5 * dt * (sigma**2 * i**2 + r * i)
            V_new[i] = a * V[i - 1] + b * V[i] + c * V[i + 1]

        V = V_new

    # interpolate to get value at S0
    i0 = int(S0 / dS)
    if i0 >= M:
        return V[M]
    S_low, S_high = S_vals[i0], S_vals[i0 + 1]
    V_low, V_high = V[i0], V[i0 + 1]
    return V_low + (V_high - V_low) * (S0 - S_low) / (S_high - S_low)

fd_price = fd_explicit_call(S0, K, r, sigma, T)

print("Black–Scholes price:   ", bs_price)
print("Binomial tree (N=200): ", binom_price_200)
print("FD explicit (M=200, N=4000):", fd_price)


Black–Scholes price:    10.450583572185565
Binomial tree (N=200):  10.44059125985994
FD explicit (M=200, N=4000): 10.454693721052504


In [ ]:
# Quantlib packages

In [ ]:
import QuantLib as ql

# Market/option parameters
spot_price    = 100.0
strike_price  = 100.0
risk_free     = 0.05      # flat risk-free rate
dividend_rate = 0.00      # no dividends
volatility    = 0.20      # flat vol
maturity_in_years = 1.0

# Dates and day count
calendar = ql.NullCalendar()
day_count = ql.Actual365Fixed()
today = ql.Date.todaysDate()
ql.Settings.instance().evaluationDate = today

maturity_date = today + int(365 * maturity_in_years)

# Option definition
option_type = ql.Option.Call
payoff = ql.PlainVanillaPayoff(option_type, strike_price)
exercise = ql.EuropeanExercise(maturity_date)
european_option = ql.VanillaOption(payoff, exercise)


In [3]:
# Handles for spot, yield curves, and vol
spot_handle = ql.QuoteHandle(ql.SimpleQuote(spot_price))
rf_curve = ql.YieldTermStructureHandle(
    ql.FlatForward(today, risk_free, day_count)
)
div_curve = ql.YieldTermStructureHandle(
    ql.FlatForward(today, dividend_rate, day_count)
)
vol_ts = ql.BlackVolTermStructureHandle(
    ql.BlackConstantVol(today, calendar, volatility, day_count)
)

bsm_process = ql.BlackScholesMertonProcess(
    spot_handle, div_curve, rf_curve, vol_ts
)

# 1) Analytic Black–Scholes price
european_option.setPricingEngine(
    ql.AnalyticEuropeanEngine(bsm_process)
)
bs_price = european_option.NPV()
print("Analytic BS price:", bs_price)


Analytic BS price: 10.450583572185575


In [4]:
# 2) Binomial tree price (CRR, 200 steps)
steps = 200
binom_engine = ql.BinomialVanillaEngine(bsm_process, "crr", steps)
european_option.setPricingEngine(binom_engine)
binom_price = european_option.NPV()
print("Binomial (CRR, 200 steps):", binom_price)


Binomial (CRR, 200 steps): 10.440278278758278


In [5]:
# 3) Finite-difference price (FD Black–Scholes)
tGrid = 200      # time steps
xGrid = 200      # space steps
fd_engine = ql.FdBlackScholesVanillaEngine(
    bsm_process, tGrid, xGrid
)
european_option.setPricingEngine(fd_engine)
fd_price = european_option.NPV()
print("FD Black–Scholes price:", fd_price)


FD Black–Scholes price: 10.452156734646318
